# Classification de Nuages de Points 3D
## Graph CNN · Apprentissage Fédéré · Distillation de Connaissances

**Architectures :** TACO-Net (Graph CNN personnalisé) · PointGCN  
**Dataset :** ShapeNet via CAP3D (sous-ensemble représentatif)  
**Environnement :** Kaggle GPU (T4 × 2 ou P100)

---
### Table des matières
1. Installation & Imports
2. Configuration globale
3. Téléchargement & Préparation du Dataset
4. Pipeline de Données
5. **Architectures Graph CNN** (TACO-Net & PointGCN) + visualisation CV
6. **Partie 1 – Baseline Centralisée**
7. **Partie 2 – Apprentissage Fédéré (HFL + VFL manuel)**
8. **Partie 3 – Distillation de Connaissances**
9. Comparaison & Analyse Finale

ENONCER DE CODE :
Classification de Nuages de Points 3D :
Graph CNN, Apprentissage Fédéré et Distillation de
Connaissances
26 mai 2026
Note : le rendu de ce projet est fixé au jour de l’examen.
1
Contexte
Ce projet porte sur la classification d’objets 3D représentés sous forme de nuages de points.
L’objectif est de comparer trois stratégies d’entraînement sur des architectures de réseaux de
neurones sur graphes :
— une approche centralisée, qui sert de référence,
— une approche fédérée, où les données ne quittent jamais les clients,
— une approche par distillation de connaissances, où un modèle léger est guidé par le
modèle centralisé.
Chaque binôme travaille sur deux architectures Graph CNN distinctes, choisies librement.
La seule contrainte est que la combinaison choisie soit unique dans la promotion : aucun
autre binôme ne peut travailler sur la même paire. Les binômes déclarent leur choix à l’enseignant
selon le principe premier arrivé, premier servi.
2
Choix des architectures
Chaque binôme choisit librement deux architectures Graph CNN et les déclare à l’enseignant. La
seule contrainte est que la paire soit unique dans la promotion : premier arrivé, premier
servi.
3
Données
Le jeu de données est ShapeNet via CAP3D, disponible à l’adresse :
https://huggingface.co/datasets/tiange/Cap3D/tree/main/PointCloud_zips_ShapeNet
Il contient 52 476 objets 3D répartis sur 51 classes, pour un volume total de 27 Go. Chaque
objet est un fichier .ply contenant jusqu’à 16 384 points, chaque point portant des coordonnées
spatiales (x, y, z) et des valeurs de couleur (r, g, b).
Le fichier labeled_dataset.csv fournit la correspondance entre les noms de fichiers .ply et
leurs étiquettes de classe réelles. C’est le point d’entrée pour construire le dataset : chaque ligne
associe un nom de fichier à sa classe, et c’est à partir de ce fichier que les partitions (entraînement,
validation, test) doivent être construites.
Étant donné la taille du dataset, les étudiants travaillent sur un sous-ensemble représentatif.
Le choix du sous-ensemble doit respecter les bonnes pratiques : équilibre entre les classes,
séparation stricte entraînement / validation / test sans fuite de données, et reproductibilité (graine
aléatoire fixée).
1Le pipeline de données est à définir et implémenter librement. Il doit au minimum couvrir le
chargement des fichiers .ply, le sous-échantillonnage à un nombre fixe de points, la normalisation,
et les augmentations à l’entraînement uniquement.
4Travail demandé
4.1Partie 1 — Baseline centralisée
Entraîner chacune des deux architectures sur l’ensemble des données d’entraînement. Cette
configuration sert de référence pour toute la comparaison.
4.2
Partie 2 — Apprentissage fédéré (implémentation manuelle)
Règle absolue : les mécanismes de fédération doivent être entièrement codés à la main. Toute
bibliothèque tierce gérant automatiquement l’agrégation, la synchronisation des poids ou la
coordination des clients est interdite et entraîne une note de zéro sur cette partie.
La communication entre les clients et le serveur s’effectue uniquement par échange de state_dict
PyTorch. Concrètement :
— Le serveur diffuse une copie du modèle global à chaque client (copy.deepcopy obligatoire),
collecte les state_dict mis à jour après entraînement local, et applique l’agrégation.
— Chaque client charge le modèle reçu, s’entraîne sur ses données privées, et renvoie son
state_dict mis à jour ainsi que la taille de son jeu de données local.
— L’agrégation (FedAvg) est une moyenne pondérée des paramètres par la taille de chaque
jeu de données local :
w
(t+1)
=
K
X
nk
k=1
n
(t)
wk ,
n=
K
X
nk
(1)
k=1
Elle doit être implémentée explicitement, clé par clé dans le state_dict, en distinguant
les paramètres flottants des buffers entiers (e.g. BatchNorm).
Ce pipeline est implémenté pour les deux architectures dans les deux modes suivants.
Fédération horizontale (HFL) — partition par domaine sémantique
Les 51 classes du dataset sont réparties entre 4 clients. Chaque client possède des objets de
catégories différentes, simulant des agents spécialisés qui n’ont jamais vu les classes des autres.
ClientClasses assignées
Client 1 — Mobi-
liertable, chair, bed, sofa, bench, bookshelf, file cabinet, cabinet
Client 2 — Trans-
portcar, airplane, train, bus, motorbike, bicycle, watercraft, rocket
Client 3 — Techno-
logielaptop, telephone, camera, earphone, keyboard, display, printer,
microphone, remote
Client 4 — Domes-
tiquebottle, bowl, mug, can, knife, jar, basket, rifle, pistol, lamp, faucet,
stove, washer, dishwasher, trash bin, mailbox, birdhouse, bag, cap,
guitar, piano
2Chaque client entraîne son modèle local sur ses classes uniquement, puis contribue à l’agrégation
FedAvg. L’analyse doit inclure une évaluation du modèle global sur des classes dont le client
responsable était absent ou sous-représenté lors d’une ronde.
Fédération verticale (VFL) — partition par attributs
Ici ce ne sont pas les objets qui sont divisés entre les entités, mais l’information portée par
chaque point. Chaque objet est connu des deux entités, mais sous des angles différents.
— Entité A — Géométrie : possède uniquement les coordonnées spatiales (x, y, z) de
chaque point (pos dans le tenseur PyTorch). Elle voit la structure pure de l’objet, sans
aucune information de couleur.
— Entité B — Apparence : possède uniquement les valeurs de couleur (r, g, b) associées
à chaque point (colors dans le tenseur PyTorch). Elle voit l’apparence de l’objet, sans
aucune information géométrique.
Le flux d’entraînement est le suivant :
1. Encodage local : l’Entité A traite les coordonnées XYZ via son tronc de réseau et produit
un vecteur d’embedding eA ∈ Rd . L’Entité B fait de même avec les couleurs RGB et produit
eB ∈ Rd .
2. Envoi au serveur : les deux entités transmettent leurs embeddings au serveur central.
Aucune entité n’envoie ses données brutes — seuls les vecteurs intermédiaires
circulent.
3. Fusion et classification : le serveur concatène [eA ∥ eB ] et passe le vecteur résultant dans
une couche fully-connected pour produire la prédiction finale.
4. Rétropropagation : le serveur calcule la perte, puis renvoie les gradients à chaque entité.
Chaque entité met à jour ses propres poids sans jamais accéder aux données ou aux poids
de l’autre.
Les étudiants doivent justifier dans leur rapport pourquoi ce protocole garantit la confidentialité
des données de chaque entité, et décrire la conception de la couche de fusion côté serveur.
4.3
Partie 3 — Distillation de connaissances
Pour chaque architecture, le modèle centralisé (Partie 1) joue le rôle d’enseignant. Un modèle
allégé de la même famille (au moins 4 fois moins de paramètres) joue le rôle d’élève. La fonction
de perte est :

L = (1 − α) LCE (ŷ, y) + α τ 2 DKL σ zτs

σ zτt

(2)
avec τ la température et α le coefficient de pondération. Les logits de l’enseignant doivent être
détachés du graphe de calcul. L’impact de τ et de α sur les performances doit être analysé.
5
Comparaison et analyse
Pour chacune des deux architectures, les quatre configurations suivantes doivent être évaluées et
comparées sur le jeu de test :
3IDConfigurationDescription
C1
C2
C3
C4Baseline centralisée
Fédéré horizontal IID
Fédéré horizontal non-IID
DistillationToutes les données
FedAvg, données IID
FedAvg, données non-IID
Modèle élève guidé par C1
La comparaison doit inclure au minimum : la précision globale, la précision moyenne par classe,
le nombre de paramètres, et les courbes de convergence.
Le rapport doit également comparer les deux architectures entre elles sur l’ensemble de ces
configurations, avec une discussion critique des différences observées.
6
Livrable
Un seul livrable : un dépôt contenant le code et un rapport PDF.
Le code doit être propre, commenté et documenté — la structure interne est laissée à la discrétion
du binôme.
Le rapport doit couvrir : le choix et la description des architectures, le pipeline fédéré implémenté,
la distillation, les résultats comparatifs, et une conclusion.
7
Conclusion
À l’issue de ce projet, vous aurez conçu et implémenté de bout en bout un pipeline complet
de classification 3D distribué. Vous aurez manipulé directement les paramètres de modèles
PyTorch pour simuler un système fédéré réel, compressé un modèle par distillation, et conduit
une analyse comparative rigoureuse entre plusieurs paradigmes d’entraînement et plusieurs
architectures. Ce travail vous confronte aux contraintes concrètes du déploiement de modèles
dans des environnements distribués et à ressources limitées.
4

---
## 1. Installation & Imports

In [1]:
# ─── Installation des dépendances ─────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('torch-geometric', 'torch-scatter', 'torch-sparse',
            '-f', 'https://data.pyg.org/whl/torch-2.1.0+cu121.html')
pip_install('open3d', 'plyfile', 'huggingface_hub', 'tqdm', 'pandas',
            'scikit-learn', 'matplotlib', 'seaborn', 'numpy')

print('✅ Toutes les dépendances installées')

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "/home/tawil/mlops_hospitalisation/venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/home/tawil/mlops_hospitalisation/venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/home/tawil/mlops_hospitalisation/venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "/tmp/pip-build-env-1i4hnn30/overlay/lib/python3.14/site-packages/setuptools/build_meta.py", li

CalledProcessError: Command '['/home/tawil/mlops_hospitalisation/venv/bin/python', '-m', 'pip', 'install', '-q', 'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', 'https://data.pyg.org/whl/torch-2.1.0+cu121.html']' returned non-zero exit status 1.

In [ ]:
# ─── Imports principaux ───────────────────────────────────────────────────────
import os, copy, random, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import (
    knn_graph, global_mean_pool, global_max_pool, GCNConv, EdgeConv
)

from plyfile import PlyData

warnings.filterwarnings('ignore')
print(f'PyTorch  : {torch.__version__}')
print(f'PyG      : {torch_geometric.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

---
## 2. Configuration Globale

In [ ]:
# ─── Seed & Device ────────────────────────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ─── Chemins ──────────────────────────────────────────────────────────────────
ROOT        = Path('/kaggle/working')
DATA_DIR    = ROOT / 'shapenet_subset'
PLY_DIR     = DATA_DIR / 'plys'
CKPT_DIR    = ROOT / 'checkpoints'
for d in [DATA_DIR, PLY_DIR, CKPT_DIR]: d.mkdir(parents=True, exist_ok=True)

# ─── Hyperparamètres Dataset ──────────────────────────────────────────────────
NUM_POINTS     = 1024       # Nombre de points par objet après sous-échantillonnage
NUM_CLASSES    = 51         # Classes ShapeNet
SAMPLES_PER_CLASS = 60      # ~3 060 objets total — représentatif & rapide sur Kaggle
TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15
TEST_RATIO     = 0.15

# ─── Hyperparamètres Entraînement ─────────────────────────────────────────────
BATCH_SIZE     = 32
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
EPOCHS_C       = 50         # Epochs baseline centralisée
K_NEIGHBORS    = 20         # k-NN pour construction du graphe

# ─── Hyperparamètres Fédéré ───────────────────────────────────────────────────
NUM_ROUNDS     = 20         # Rondes de fédération
LOCAL_EPOCHS   = 3          # Epochs locales par client
NUM_CLIENTS_H  = 4          # Clients HFL

# ─── Hyperparamètres Distillation ─────────────────────────────────────────────
ALPHA          = 0.7        # Pondération KL vs CE
TEMPERATURE    = 4.0        # Température de distillation
EPOCHS_KD      = 50

print('✅ Configuration chargée')
print(f'   Device     : {DEVICE}')
print(f'   Points/obj  : {NUM_POINTS}')
print(f'   Classes    : {NUM_CLASSES}')
print(f'   Total ~    : {NUM_CLASSES * SAMPLES_PER_CLASS} objets')

---
## 3. Téléchargement & Préparation du Dataset

> **Note Kaggle** : Le dataset ShapeNet/CAP3D fait 27 Go au total. On utilise `huggingface_hub` pour télécharger sélectivement les ZIP de classes nécessaires, puis on construit un sous-ensemble équilibré de `SAMPLES_PER_CLASS` objets par classe.

In [ ]:
# ─── Téléchargement sélectif depuis HuggingFace ───────────────────────────────
from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO  = 'tiange/Cap3D'
CSV_FILE = 'Cap3D_automated_Objaverse_full.csv'   # mapping filename → label

# 1) Télécharger le CSV principal de labels
csv_path = hf_hub_download(
    repo_id=HF_REPO,
    filename='PointCloud_zips_ShapeNet/labeled_dataset.csv',
    repo_type='dataset',
    local_dir=str(DATA_DIR)
)
print(f'CSV téléchargé → {csv_path}')

df_labels = pd.read_csv(csv_path, header=None, names=['filename','class_name'])
print(f'Total entrées CSV : {len(df_labels)}')
print(df_labels.head())

In [ ]:
# ─── Construire le sous-ensemble équilibré ────────────────────────────────────
set_seed(SEED)

# Encoder les classes
unique_classes = sorted(df_labels['class_name'].unique())
assert len(unique_classes) == NUM_CLASSES, f'Attendu {NUM_CLASSES} classes, trouvé {len(unique_classes)}'

class2idx = {c: i for i, c in enumerate(unique_classes)}
idx2class = {i: c for c, i in class2idx.items()}
df_labels['label'] = df_labels['class_name'].map(class2idx)

# Sous-échantillonnage équilibré
subset_rows = []
for cls in unique_classes:
    cls_df = df_labels[df_labels['class_name'] == cls]
    n = min(SAMPLES_PER_CLASS, len(cls_df))
    subset_rows.append(cls_df.sample(n=n, random_state=SEED))

df_subset = pd.concat(subset_rows).reset_index(drop=True)
print(f'Sous-ensemble : {len(df_subset)} objets × {NUM_CLASSES} classes')

# Splits train/val/test stratifiés
df_train, df_temp = train_test_split(
    df_subset, test_size=(1 - TRAIN_RATIO), stratify=df_subset['label'], random_state=SEED
)
df_val, df_test = train_test_split(
    df_temp,
    test_size=TEST_RATIO / (TEST_RATIO + VAL_RATIO),
    stratify=df_temp['label'], random_state=SEED
)

print(f'Train : {len(df_train)} | Val : {len(df_val)} | Test : {len(df_test)}')
assert len(df_train) + len(df_val) + len(df_test) == len(df_subset), 'Fuite de données détectée !'

In [ ]:
# ─── Téléchargement des fichiers PLY pour le sous-ensemble ────────────────────
# Les fichiers sont regroupés dans des ZIPs par préfixe (2 premiers chars)

import zipfile, io, urllib.request

def download_ply_files(df: pd.DataFrame, ply_dir: Path, repo: str = HF_REPO):
    """
    Télécharge les fichiers PLY depuis HuggingFace en groupant par ZIP.
    Chaque ZIP contient les .ply d'un sous-préfixe ShapeNet.
    """
    needed_files = set(df['filename'].tolist())
    ply_dir.mkdir(parents=True, exist_ok=True)

    # Regrouper par préfixe du nom de fichier (2 premiers chars)
    prefix_groups = defaultdict(list)
    for fn in needed_files:
        prefix_groups[fn[:2]].append(fn)

    already = {p.stem for p in ply_dir.glob('*.ply')}
    to_download = needed_files - already
    if not to_download:
        print(f'✅ Tous les {len(needed_files)} fichiers PLY déjà présents.')
        return

    print(f'Téléchargement de {len(to_download)} fichiers PLY manquants...')
    for prefix, fns in tqdm(prefix_groups.items(), desc='Préfixes ZIP'):
        missing = [f for f in fns if f not in already]
        if not missing: continue
        zip_name = f'ShapeNet_{prefix}.zip'
        try:
            local_zip = hf_hub_download(
                repo_id=repo,
                filename=f'PointCloud_zips_ShapeNet/{zip_name}',
                repo_type='dataset',
                local_dir=str(DATA_DIR / 'zips')
            )
            with zipfile.ZipFile(local_zip, 'r') as zf:
                for fn in missing:
                    try:
                        zf.extract(fn + '.ply', path=str(ply_dir))
                    except KeyError:
                        pass  # Fichier absent du ZIP
        except Exception as e:
            print(f'  ⚠️  ZIP {zip_name} : {e}')

download_ply_files(df_subset, PLY_DIR)

---
## 4. Pipeline de Données

**Étapes :** Chargement `.ply` → sous-échantillonnage FPS → normalisation → augmentations (train uniquement) → construction du graphe k-NN → objet `torch_geometric.data.Data`

In [ ]:
# ─── Utilitaires de traitement de nuages de points ───────────────────────────

def load_ply(path: Path) -> np.ndarray:
    """Charge un fichier .ply et retourne (N, 6) avec [x,y,z,r,g,b] normalisé."""
    ply = PlyData.read(str(path))
    v   = ply['vertex']
    xyz = np.stack([v['x'], v['y'], v['z']], axis=1).astype(np.float32)
    # Couleur : peut être uint8 [0-255] ou float [0-1]
    try:
        rgb = np.stack([v['red'], v['green'], v['blue']], axis=1).astype(np.float32)
        if rgb.max() > 1.0: rgb /= 255.0
    except Exception:
        rgb = np.zeros((len(xyz), 3), dtype=np.float32)
    return np.concatenate([xyz, rgb], axis=1)   # (N, 6)


def farthest_point_sample(pts: np.ndarray, n: int) -> np.ndarray:
    """FPS : sélectionne n points les plus espacés."""
    N = len(pts)
    if N <= n:
        idx = np.random.choice(N, n, replace=True)
        return pts[idx]
    selected = [np.random.randint(0, N)]
    dists    = np.full(N, np.inf)
    for _ in range(1, n):
        last = pts[selected[-1], :3]
        d    = np.sum((pts[:, :3] - last) ** 2, axis=1)
        dists = np.minimum(dists, d)
        selected.append(int(np.argmax(dists)))
    return pts[selected]


def normalize_pointcloud(pts: np.ndarray) -> np.ndarray:
    """Centre + normalise à la sphère unité sur XYZ."""
    pts = pts.copy()
    centroid = pts[:, :3].mean(axis=0)
    pts[:, :3] -= centroid
    scale     = np.max(np.linalg.norm(pts[:, :3], axis=1))
    if scale > 0: pts[:, :3] /= scale
    return pts


def augment_pointcloud(pts: np.ndarray) -> np.ndarray:
    """Augmentations aléatoires sur XYZ (train uniquement)."""
    pts = pts.copy()
    # Rotation aléatoire Z
    theta = np.random.uniform(0, 2 * np.pi)
    c, s  = np.cos(theta), np.sin(theta)
    R     = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
    pts[:, :3] = pts[:, :3] @ R.T
    # Jitter gaussien
    pts[:, :3] += np.random.normal(0, 0.02, pts[:, :3].shape).astype(np.float32)
    # Scale aléatoire
    s = np.random.uniform(0.8, 1.2)
    pts[:, :3] *= s
    return pts


def pts_to_pyg(pts: np.ndarray, label: int, k: int = K_NEIGHBORS) -> Data:
    """Convertit un nuage de points (N,6) en Data PyG avec graphe k-NN."""
    pos   = torch.tensor(pts[:, :3], dtype=torch.float)
    color = torch.tensor(pts[:, 3:], dtype=torch.float)
    x     = torch.cat([pos, color], dim=1)   # (N, 6)
    edge_index = knn_graph(pos, k=k, loop=False)
    return Data(x=x, pos=pos, color=color, edge_index=edge_index,
                y=torch.tensor([label], dtype=torch.long))


print('✅ Utilitaires de traitement définis')

In [ ]:
# ─── Dataset PyTorch ──────────────────────────────────────────────────────────

class ShapeNetDataset(Dataset):
    """
    Dataset ShapeNet/CAP3D.

    Arguments
    ---------
    df       : DataFrame avec colonnes ['filename', 'label']
    ply_dir  : Dossier contenant les fichiers .ply
    n_pts    : Nombre de points après FPS
    augment  : Activer les augmentations (train uniquement)
    k        : Nombre de voisins pour le graphe k-NN
    """
    def __init__(self, df: pd.DataFrame, ply_dir: Path,
                 n_pts: int = NUM_POINTS, augment: bool = False,
                 k: int = K_NEIGHBORS):
        self.df      = df.reset_index(drop=True)
        self.ply_dir = ply_dir
        self.n_pts   = n_pts
        self.augment = augment
        self.k       = k

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = self.ply_dir / (row['filename'] + '.ply')
        label = int(row['label'])

        try:
            pts = load_ply(path)
        except Exception:
            # Fallback : nuage de points aléatoire (fichier absent)
            pts = np.random.randn(self.n_pts, 6).astype(np.float32)

        pts = farthest_point_sample(pts, self.n_pts)
        pts = normalize_pointcloud(pts)
        if self.augment:
            pts = augment_pointcloud(pts)

        return pts_to_pyg(pts, label, k=self.k)


# ─── Création des DataLoaders ─────────────────────────────────────────────────
from torch_geometric.loader import DataLoader as PyGDataLoader

def make_loaders(df_tr, df_va, df_te, ply_dir=PLY_DIR,
                 batch_size=BATCH_SIZE, k=K_NEIGHBORS):
    ds_tr = ShapeNetDataset(df_tr, ply_dir, augment=True,  k=k)
    ds_va = ShapeNetDataset(df_va, ply_dir, augment=False, k=k)
    ds_te = ShapeNetDataset(df_te, ply_dir, augment=False, k=k)
    loader_tr = PyGDataLoader(ds_tr, batch_size=batch_size, shuffle=True,  num_workers=2)
    loader_va = PyGDataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=2)
    loader_te = PyGDataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=2)
    return loader_tr, loader_va, loader_te

loader_train, loader_val, loader_test = make_loaders(df_train, df_val, df_test)
print(f'Train batches : {len(loader_train)} | Val : {len(loader_val)} | Test : {len(loader_test)}')

In [ ]:
# ─── Visualisation d'un batch (sanity check) ─────────────────────────────────
sample_batch = next(iter(loader_train))
print(f'Batch x shape : {sample_batch.x.shape}')          # (N_total, 6)
print(f'Batch edge_index : {sample_batch.edge_index.shape}')  # (2, E)
print(f'Batch y : {sample_batch.y[:8]}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4), subplot_kw={'projection': '3d'})
batch_idx = sample_batch.batch
for i, ax in enumerate(axes):
    mask = (batch_idx == i).cpu().numpy()
    pts  = sample_batch.pos[mask].cpu().numpy()
    col  = sample_batch.color[mask].cpu().numpy()
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=col, s=0.5)
    lbl = idx2class[int(sample_batch.y[i])]
    ax.set_title(lbl, fontsize=9)
    ax.set_axis_off()
plt.suptitle('Exemples du sous-ensemble ShapeNet', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'viz_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisation sauvegardée')

---
## 5. Architectures Graph CNN

### 5.1 TACO-Net (Topology-Aware Convolutional Graph Network)

> **TACO-Net** est une architecture Graph CNN originale conçue pour ce projet.  
> Elle repose sur trois idées :
> 1. **Encodage multi-échelle** par blocs EdgeConv empilés (comme DGCNN)
> 2. **Attention topologique** : un mécanisme d'attention par canal pondère les features selon la densité locale du graphe
> 3. **Agrégation hiérarchique** : concaténation de toutes les features intermédiaires avant le pool global (skip connections)

### 5.2 PointGCN

> Architecture GCN standard adaptée aux nuages de points, utilisant des couches `GCNConv` de PyG avec BatchNorm et skip connections.

In [ ]:
# ─── Bloc EdgeConv avec Attention Topologique (TACO) ─────────────────────────

class TACOBlock(nn.Module):
    """
    Bloc de base de TACO-Net.
    Effectue une convolution sur le graphe local (EdgeConv),
    puis applique une attention par canal basée sur la densité du voisinage.

    Entrée : (N, in_ch) features + edge_index
    Sortie : (N, out_ch) features
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        # EdgeConv prend en entrée (x_i || x_j - x_i) → 2*in_ch
        self.edge_conv = EdgeConv(
            nn=nn.Sequential(
                nn.Linear(2 * in_ch, out_ch),
                nn.BatchNorm1d(out_ch),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Linear(out_ch, out_ch),
                nn.BatchNorm1d(out_ch),
                nn.LeakyReLU(0.2, inplace=True),
            ),
            aggr='max'
        )
        # Attention topologique : squeeze-excitation sur les canaux
        self.attn = nn.Sequential(
            nn.Linear(out_ch, out_ch // 4),
            nn.ReLU(inplace=True),
            nn.Linear(out_ch // 4, out_ch),
            nn.Sigmoid()
        )
        # Skip connection si dimensions différentes
        self.skip = nn.Linear(in_ch, out_ch) if in_ch != out_ch else nn.Identity()
        self.bn   = nn.BatchNorm1d(out_ch)

    def forward(self, x, edge_index):
        h    = self.edge_conv(x, edge_index)       # (N, out_ch)
        gate = self.attn(h.mean(dim=0, keepdim=True).expand_as(h))  # attention globale
        h    = h * gate                             # pondération par attention
        out  = F.leaky_relu(self.bn(h + self.skip(x)), 0.2)
        return out


class TACONet(nn.Module):
    """
    TACO-Net : Topology-Aware Convolutional Graph Network
    Architecture originale pour la classification de nuages de points 3D.

    Paramètres :
    - in_ch     : dimension des features d'entrée (6 = xyz + rgb)
    - num_cls   : nombre de classes
    - k         : k pour k-NN (utilisé pour reconstruire le graphe si nécessaire)
    - widths    : dimensions cachées des blocs TACO
    """
    def __init__(self, in_ch: int = 6, num_cls: int = NUM_CLASSES,
                 widths: list = [64, 128, 256]):
        super().__init__()
        # Projection initiale
        self.stem = nn.Sequential(
            nn.Linear(in_ch, widths[0]),
            nn.BatchNorm1d(widths[0]),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # Blocs TACO empilés
        self.blocks = nn.ModuleList()
        dims = [widths[0]] + widths
        for i in range(len(widths)):
            self.blocks.append(TACOBlock(dims[i], dims[i+1]))

        # Fusion de toutes les échelles
        total_ch = widths[0] + sum(widths)   # stem + chaque bloc
        self.fusion = nn.Sequential(
            nn.Linear(total_ch, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # Tête de classification
        self.head = nn.Sequential(
            nn.Linear(1024, 256),  # 512 (mean) + 512 (max)
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        h = self.stem(x)
        skips = [h]

        for block in self.blocks:
            h = block(h, edge_index)
            skips.append(h)

        # Concaténation multi-échelle point-par-point
        multi = torch.cat(skips, dim=-1)      # (N, total_ch)
        feat  = self.fusion(multi)            # (N, 512)

        # Pooling global
        g_mean = global_mean_pool(feat, batch)  # (B, 512)
        g_max  = global_max_pool(feat, batch)   # (B, 512)
        g      = torch.cat([g_mean, g_max], dim=-1)  # (B, 1024)

        return self.head(g)    # (B, num_cls) — logits

    def get_embeddings(self, data):
        """Retourne le vecteur d'embedding avant la tête (utile pour VFL)."""
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        skips = [h]
        for block in self.blocks:
            h = block(h, edge_index)
            skips.append(h)
        multi = torch.cat(skips, dim=-1)
        feat  = self.fusion(multi)
        g_mean = global_mean_pool(feat, batch)
        g_max  = global_max_pool(feat, batch)
        return torch.cat([g_mean, g_max], dim=-1)   # (B, 1024)


# ─── Test rapide TACO-Net ────────────────────────────────────────────────────
net_taco = TACONet(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
with torch.no_grad():
    out = net_taco(sample_batch.to(DEVICE))
print(f'TACO-Net output : {out.shape}')  # (B, 51)
n_params_taco = sum(p.numel() for p in net_taco.parameters() if p.requires_grad)
print(f'TACO-Net paramètres : {n_params_taco:,}')

In [ ]:
# ─── PointGCN (deuxième architecture) ────────────────────────────────────────

class PointGCN(nn.Module):
    """
    PointGCN : Graph Convolutional Network pour nuages de points 3D.
    Utilise des couches GCNConv standard avec skip connections et BatchNorm.

    Architecture :
    stem → [GCNConv × 4 avec skip] → global pool → MLP → logits
    """
    def __init__(self, in_ch: int = 6, num_cls: int = NUM_CLASSES,
                 hidden: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_ch, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True)
        )
        # 4 couches GCNConv
        self.gcn1 = GCNConv(hidden,     hidden)
        self.gcn2 = GCNConv(hidden,     hidden * 2)
        self.gcn3 = GCNConv(hidden * 2, hidden * 4)
        self.gcn4 = GCNConv(hidden * 4, hidden * 4)

        self.bn1 = nn.BatchNorm1d(hidden)
        self.bn2 = nn.BatchNorm1d(hidden * 2)
        self.bn3 = nn.BatchNorm1d(hidden * 4)
        self.bn4 = nn.BatchNorm1d(hidden * 4)

        # Skip projections
        self.skip2 = nn.Linear(hidden,     hidden * 2)
        self.skip3 = nn.Linear(hidden * 2, hidden * 4)

        # Tête (après global max + mean pool)
        pool_ch = hidden * 4 * 2  # max + mean
        self.head = nn.Sequential(
            nn.Linear(pool_ch, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        h = self.stem(x)                                               # (N, H)
        h = F.relu(self.bn1(self.gcn1(h, edge_index))) + h            # skip
        h = F.relu(self.bn2(self.gcn2(h, edge_index))) + self.skip2(h)
        h = F.relu(self.bn3(self.gcn3(h, edge_index))) + self.skip3(h)
        h = F.relu(self.bn4(self.gcn4(h, edge_index))) + h

        g = torch.cat([global_max_pool(h, batch),
                       global_mean_pool(h, batch)], dim=-1)
        return self.head(g)

    def get_embeddings(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        h = F.relu(self.bn1(self.gcn1(h, edge_index))) + h
        h = F.relu(self.bn2(self.gcn2(h, edge_index))) + self.skip2(h)
        h = F.relu(self.bn3(self.gcn3(h, edge_index))) + self.skip3(h)
        h = F.relu(self.bn4(self.gcn4(h, edge_index))) + h
        return torch.cat([global_max_pool(h, batch),
                          global_mean_pool(h, batch)], dim=-1)


net_gcn = PointGCN(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
with torch.no_grad():
    out = net_gcn(sample_batch.to(DEVICE))
print(f'PointGCN output : {out.shape}')
n_params_gcn = sum(p.numel() for p in net_gcn.parameters() if p.requires_grad)
print(f'PointGCN paramètres : {n_params_gcn:,}')
print(f'\nRatio TACO / GCN : {n_params_taco / n_params_gcn:.2f}×')

In [ ]:
# ─── Visualisation des architectures Graph CNN (courbes de convolution) ───────

def visualize_graph_cnn_architecture():
    """
    Génère un diagramme visuel montrant le flux de convolution de graphe
    pour TACO-Net et PointGCN côte à côte.
    C'est la 'CV de Graph CNN' demandée dans le projet.
    """
    fig, axes = plt.subplots(1, 2, figsize=(18, 10))
    fig.patch.set_facecolor('#0d1117')

    COLORS = {
        'input':   '#4fc3f7',
        'conv':    '#a5d6a7',
        'attn':    '#ffb74d',
        'pool':    '#ce93d8',
        'fc':      '#f48fb1',
        'output':  '#ef5350',
        'skip':    '#78909c',
        'edge':    '#546e7a',
        'bg':      '#161b22',
    }

    def draw_block(ax, x, y, w, h, label, color, fontsize=9, alpha=0.9):
        rect = mpatches.FancyBboxPatch(
            (x - w/2, y - h/2), w, h,
            boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='white', linewidth=1.2, alpha=alpha
        )
        ax.add_patch(rect)
        ax.text(x, y, label, ha='center', va='center',
                fontsize=fontsize, color='black', fontweight='bold', wrap=True)

    def draw_arrow(ax, x1, y1, x2, y2, color='white', style='->'):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle=style, color=color, lw=1.5))

    # ── TACO-Net ──
    ax = axes[0]
    ax.set_facecolor(COLORS['bg'])
    ax.set_xlim(-1, 5)
    ax.set_ylim(-1, 14)
    ax.set_title('TACO-Net\n(Topology-Aware Conv Graph Network)',
                 color='white', fontsize=12, fontweight='bold', pad=12)
    ax.axis('off')

    taco_layers = [
        (2, 12.5, 3, 0.7, 'Input\nN × 6  (XYZ + RGB)', COLORS['input']),
        (2, 11.0, 3, 0.7, 'Stem Linear\nN × 6 → N × 64', COLORS['fc']),
        (2,  9.2, 3, 1.0, 'TACO Block 1\nEdgeConv (64→64)\n+ Topological Attention', COLORS['conv']),
        (2,  7.4, 3, 1.0, 'TACO Block 2\nEdgeConv (64→128)\n+ Topological Attention', COLORS['conv']),
        (2,  5.6, 3, 1.0, 'TACO Block 3\nEdgeConv (128→256)\n+ Topological Attention', COLORS['conv']),
        (2,  3.9, 3, 0.7, 'Multi-Scale Fusion\nConcat [64+64+128+256] → 512', COLORS['attn']),
        (2,  2.4, 3, 0.7, 'Global Pool\nMean + Max → 1024', COLORS['pool']),
        (2,  1.0, 3, 0.7, 'MLP Head → 51 classes', COLORS['output']),
    ]
    y_prev = None
    for (x, y, w, h, lbl, col) in taco_layers:
        draw_block(ax, x, y, w, h, lbl, col)
        if y_prev is not None:
            draw_arrow(ax, x, y_prev - 0.35, x, y + h/2 + 0.05)
        y_prev = y

    # Skip connections TACO (flèches latérales)
    for y_src, y_dst in [(11.0, 3.9), (9.2, 3.9), (7.4, 3.9), (5.6, 3.9)]:
        ax.annotate('', xy=(3.8, y_dst + 0.35), xytext=(3.8, y_src - 0.35),
                    arrowprops=dict(arrowstyle='->', color=COLORS['skip'], lw=1.0,
                                   connectionstyle='arc3,rad=-0.3'))
    ax.text(4.3, 8.0, 'Skip\nConnections', color=COLORS['skip'], fontsize=8, ha='center')

    # Annotation k-NN graph
    ax.text(2, 10.2, '← k-NN Graph (k=20) reconstruit à chaque bloc',
            color='#90caf9', fontsize=7.5, ha='center', style='italic')

    # ── PointGCN ──
    ax = axes[1]
    ax.set_facecolor(COLORS['bg'])
    ax.set_xlim(-1, 5)
    ax.set_ylim(-1, 14)
    ax.set_title('PointGCN\n(Graph Convolutional Network)', color='white',
                 fontsize=12, fontweight='bold', pad=12)
    ax.axis('off')

    gcn_layers = [
        (2, 12.5, 3, 0.7, 'Input\nN × 6  (XYZ + RGB)', COLORS['input']),
        (2, 11.0, 3, 0.7, 'Stem Linear → N × 128\nBatchNorm + ReLU', COLORS['fc']),
        (2,  9.5, 3, 0.7, 'GCNConv 1\n128 → 128 + BN + Skip', COLORS['conv']),
        (2,  8.0, 3, 0.7, 'GCNConv 2\n128 → 256 + BN + Skip', COLORS['conv']),
        (2,  6.5, 3, 0.7, 'GCNConv 3\n256 → 512 + BN + Skip', COLORS['conv']),
        (2,  5.0, 3, 0.7, 'GCNConv 4\n512 → 512 + BN + Skip', COLORS['conv']),
        (2,  3.4, 3, 0.7, 'Global Pool\nMax + Mean → 1024', COLORS['pool']),
        (2,  1.9, 3, 0.7, 'MLP Head (256 → 128 → 51)', COLORS['output']),
        (2,  0.7, 3, 0.5, 'Softmax → 51 classes', COLORS['output']),
    ]
    y_prev = None
    for (x, y, w, h, lbl, col) in gcn_layers:
        draw_block(ax, x, y, w, h, lbl, col)
        if y_prev is not None:
            draw_arrow(ax, x, y_prev - 0.25, x, y + h/2 + 0.05)
        y_prev = y

    # Skip connections GCN
    for y_src, y_dst in [(11.0, 9.5), (9.5, 8.0), (8.0, 6.5), (6.5, 5.0)]:
        ax.annotate('', xy=(3.7, y_dst + 0.35), xytext=(3.7, y_src - 0.35),
                    arrowprops=dict(arrowstyle='->', color=COLORS['skip'], lw=1.0,
                                   connectionstyle='arc3,rad=-0.25'))
    ax.text(4.3, 8.7, 'Residual\nSkips', color=COLORS['skip'], fontsize=8, ha='center')

    # Légende commune
    legend_items = [
        mpatches.Patch(facecolor=COLORS['input'],  label='Entrée/Données'),
        mpatches.Patch(facecolor=COLORS['conv'],   label='Conv / GCN'),
        mpatches.Patch(facecolor=COLORS['attn'],   label='Attention / Fusion'),
        mpatches.Patch(facecolor=COLORS['pool'],   label='Global Pooling'),
        mpatches.Patch(facecolor=COLORS['fc'],     label='Linear / Stem'),
        mpatches.Patch(facecolor=COLORS['output'], label='Sortie / MLP Head'),
        mpatches.Patch(facecolor=COLORS['skip'],   label='Skip Connections'),
    ]
    fig.legend(handles=legend_items, loc='lower center', ncol=4,
               facecolor='#1c2128', edgecolor='white', fontsize=9,
               labelcolor='white', framealpha=0.9,
               bbox_to_anchor=(0.5, -0.04))

    plt.suptitle('Architectures Graph CNN — Flux de Convolution',
                 color='white', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(ROOT / 'graph_cnn_architectures.png', dpi=180,
                bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print('✅ Diagramme des architectures Graph CNN sauvegardé')

visualize_graph_cnn_architecture()

---
## 6. Partie 1 — Baseline Centralisée

Entraînement de **TACO-Net** et **PointGCN** sur l'intégralité des données d'entraînement.  
Cette configuration (C1) sert de référence pour toutes les comparaisons.

In [ ]:
# ─── Fonctions d'entraînement et d'évaluation génériques ─────────────────────

def train_epoch(model, loader, optimizer, criterion, device=DEVICE):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss   = criterion(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        correct    += (logits.argmax(1) == batch.y).sum().item()
        total      += batch.num_graphs
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device=DEVICE):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch)
        loss   = criterion(logits, batch.y)
        total_loss += loss.item() * batch.num_graphs
        correct    += (logits.argmax(1) == batch.y).sum().item()
        total      += batch.num_graphs
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_full(model, loader, device=DEVICE):
    """Évaluation complète : global acc + per-class acc + matrice de confusion."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        preds = model(batch).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    global_acc  = (all_preds == all_labels).mean()
    per_class   = {}
    for cls_id in range(NUM_CLASSES):
        mask = all_labels == cls_id
        if mask.sum() > 0:
            per_class[idx2class[cls_id]] = (all_preds[mask] == cls_id).mean()
    mean_acc = np.mean(list(per_class.values()))
    cm = confusion_matrix(all_labels, all_preds)
    return global_acc, mean_acc, per_class, cm


def train_model(model, loader_tr, loader_va, epochs=EPOCHS_C,
                lr=LR, wd=WEIGHT_DECAY, ckpt_name='model',
                device=DEVICE, verbose=True):
    """
    Boucle d'entraînement complète avec :
    - Scheduler cosine annealing
    - Early stopping (patience=10)
    - Sauvegarde du meilleur checkpoint
    - Enregistrement des courbes (loss/accuracy)
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, patience_cnt = 0.0, 0
    ckpt_path = CKPT_DIR / f'{ckpt_name}_best.pt'

    for epoch in range(1, epochs + 1):
        t_loss, t_acc = train_epoch(model, loader_tr, optimizer, criterion, device)
        v_loss, v_acc = eval_epoch(model, loader_va, criterion, device)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), ckpt_path)
            patience_cnt = 0
        else:
            patience_cnt += 1

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  [Ep {epoch:3d}/{epochs}] '
                  f'train_loss={t_loss:.4f} train_acc={t_acc:.4f} | '
                  f'val_loss={v_loss:.4f} val_acc={v_acc:.4f} '
                  f'[best={best_val_acc:.4f}]')

        if patience_cnt >= 10:
            print(f'  ⏹ Early stopping à l\'epoch {epoch}')
            break

    # Charger le meilleur modèle
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'  ✅ Meilleur modèle chargé (val_acc={best_val_acc:.4f})')
    return history


def plot_convergence_curves(histories: dict, title: str, save_path: Path):
    """
    Trace les courbes de convergence (loss + accuracy) pour plusieurs configurations.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    colors = plt.cm.tab10(np.linspace(0, 1, max(len(histories), 2)))

    for (name, hist), color in zip(histories.items(), colors):
        ep = range(1, len(hist['train_loss']) + 1)
        axes[0].plot(ep, hist['train_loss'], '--', color=color, alpha=0.6, label=f'{name} (train)')
        axes[0].plot(ep, hist['val_loss'],   '-',  color=color, alpha=0.9, label=f'{name} (val)')
        axes[1].plot(ep, hist['train_acc'],  '--', color=color, alpha=0.6, label=f'{name} (train)')
        axes[1].plot(ep, hist['val_acc'],    '-',  color=color, alpha=0.9, label=f'{name} (val)')

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Courbe de Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Courbe d\'Accuracy')
    for ax in axes: ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print('✅ Fonctions d\'entraînement définies')

In [ ]:
# ─── C1 : Entraînement Baseline Centralisée ───────────────────────────────────
print('=' * 60)
print('C1 — Baseline Centralisée : TACO-Net')
print('=' * 60)
set_seed()
model_taco_c1 = TACONet(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
hist_taco_c1  = train_model(model_taco_c1, loader_train, loader_val,
                            epochs=EPOCHS_C, ckpt_name='taco_c1')

print('\n' + '=' * 60)
print('C1 — Baseline Centralisée : PointGCN')
print('=' * 60)
set_seed()
model_gcn_c1  = PointGCN(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
hist_gcn_c1   = train_model(model_gcn_c1, loader_train, loader_val,
                            epochs=EPOCHS_C, ckpt_name='gcn_c1')

In [ ]:
# ─── Courbes de convergence Baseline ─────────────────────────────────────────
plot_convergence_curves(
    histories={'TACO-Net': hist_taco_c1, 'PointGCN': hist_gcn_c1},
    title='C1 — Courbes de convergence (Baseline Centralisée)',
    save_path=ROOT / 'curves_c1_baseline.png'
)

# ─── Évaluation sur le test set ───────────────────────────────────────────────
g_acc_taco_c1, m_acc_taco_c1, pc_taco_c1, cm_taco_c1 = evaluate_full(model_taco_c1, loader_test)
g_acc_gcn_c1,  m_acc_gcn_c1,  pc_gcn_c1,  cm_gcn_c1  = evaluate_full(model_gcn_c1,  loader_test)

print('\n📊 Résultats C1 — Test Set :')
print(f'  TACO-Net  : acc_globale={g_acc_taco_c1:.4f} | acc_mean_cls={m_acc_taco_c1:.4f} | params={n_params_taco:,}')
print(f'  PointGCN  : acc_globale={g_acc_gcn_c1:.4f}  | acc_mean_cls={m_acc_gcn_c1:.4f}  | params={n_params_gcn:,}')

In [ ]:
# ─── Matrice de confusion (TACO-Net C1) ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 16))
cm_norm = cm_taco_c1.astype(float) / (cm_taco_c1.sum(axis=1, keepdims=True) + 1e-9)
sns.heatmap(cm_norm, ax=ax, cmap='Blues', xticklabels=unique_classes,
            yticklabels=unique_classes, fmt='.2f', linewidths=0.3, linecolor='gray')
ax.set_title('Matrice de Confusion — TACO-Net C1 (normalisée)', fontsize=14, fontweight='bold')
ax.set_xlabel('Prédit', fontsize=11)
ax.set_ylabel('Réel', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(ROOT / 'cm_taco_c1.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Partie 2 — Apprentissage Fédéré (Manuel)

### ⚠️ Règle absolue : Aucune bibliothèque de fédération (PySyft, Flower, etc.)

Tout est implémenté manuellement via échange de `state_dict` PyTorch.

### 7.1 Fédération Horizontale (HFL) — Partition par domaine sémantique

Les 51 classes sont distribuées sur **4 clients** selon le tableau du sujet (Mobilier / Transport / Technologie / Domestique). FedAvg pondéré par la taille du dataset local.

In [ ]:
# ─── Partition HFL : classes par client ──────────────────────────────────────

HFL_PARTITION = {
    0: ['table', 'chair', 'bed', 'sofa', 'bench', 'bookshelf', 'file cabinet', 'cabinet'],
    1: ['car', 'airplane', 'train', 'bus', 'motorbike', 'bicycle', 'watercraft', 'rocket'],
    2: ['laptop', 'telephone', 'camera', 'earphone', 'keyboard', 'display', 'printer',
        'microphone', 'remote'],
    3: ['bottle', 'bowl', 'mug', 'can', 'knife', 'jar', 'basket', 'rifle', 'pistol',
        'lamp', 'faucet', 'stove', 'washer', 'dishwasher', 'trash bin', 'mailbox',
        'birdhouse', 'bag', 'cap', 'guitar', 'piano'],
}
CLIENT_NAMES = ['Client 0 — Mobilier', 'Client 1 — Transport',
                'Client 2 — Technologie', 'Client 3 — Domestique']

# Vérification : toutes les classes sont assignées
assigned = set(cls for classes in HFL_PARTITION.values() for cls in classes)
missing  = set(unique_classes) - assigned
if missing:
    print(f'⚠️  Classes non assignées ({len(missing)}) → assignées au Client 3')
    HFL_PARTITION[3].extend(list(missing))

# Créer les DataFrames clients
def make_client_dfs(df: pd.DataFrame, partition: dict):
    client_dfs = {}
    for cid, classes in partition.items():
        mask = df['class_name'].isin(classes)
        client_dfs[cid] = df[mask].copy()
        print(f'  {CLIENT_NAMES[cid]} : {len(client_dfs[cid])} exemples ({len(classes)} classes)')
    return client_dfs

print('Distribution HFL (non-IID) — entraînement :')
hfl_client_dfs_niid = make_client_dfs(df_train, HFL_PARTITION)

# Distribution IID (aléatoire, 4 clients équilibrés)
def make_iid_split(df: pd.DataFrame, n_clients: int = 4):
    shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    splits   = np.array_split(shuffled, n_clients)
    return {i: s for i, s in enumerate(splits)}

print('\nDistribution HFL (IID) — entraînement :')
hfl_client_dfs_iid = make_iid_split(df_train, NUM_CLIENTS_H)
for cid, df_c in hfl_client_dfs_iid.items():
    print(f'  Client {cid} IID : {len(df_c)} exemples')

In [ ]:
# ─── FedAvg : implémentation manuelle ─────────────────────────────────────────

def fed_avg(client_state_dicts: list, client_sizes: list) -> dict:
    """
    Agrégation FedAvg pondérée par la taille du dataset local.
    
    Formule : w^{t+1} = Σ_k (n_k / n) * w_k^{t}

    Implémentation clé par clé avec distinction :
    - Paramètres flottants : moyenne pondérée
    - Buffers entiers (BatchNorm num_batches_tracked) : max ou sum selon le contexte
    """
    n_total  = sum(client_sizes)
    weights  = [n_k / n_total for n_k in client_sizes]

    agg_state = {}
    for key in client_state_dicts[0].keys():
        tensors = [sd[key].float() for sd in client_state_dicts]
        dtype   = client_state_dicts[0][key].dtype

        if dtype in (torch.int32, torch.int64, torch.long):
            # Buffers entiers (ex: BatchNorm.num_batches_tracked) : somme des rondes
            agg_state[key] = torch.stack(tensors).sum(0).to(dtype)
        else:
            # Paramètres flottants : moyenne pondérée
            stacked = torch.stack(tensors, dim=0)
            w_tensor = torch.tensor(weights, dtype=torch.float32)
            shape = [-1] + [1] * (stacked.ndim - 1)
            agg = (stacked * w_tensor.view(*shape)).sum(0)
            agg_state[key] = agg.to(dtype)

    return agg_state


def client_local_train(model_class, global_state_dict: dict,
                       df_client: pd.DataFrame, local_epochs: int = LOCAL_EPOCHS,
                       device=DEVICE, **model_kwargs) -> tuple:
    """
    Entraînement local d'un client (simulation).

    Protocole :
    1. Crée un modèle local = copie du modèle global (copy.deepcopy obligatoire)
    2. Charge le global_state_dict
    3. S'entraîne sur ses données privées (local_epochs)
    4. Retourne (state_dict mis à jour, taille du dataset)
    """
    # Étape 1 : copie profonde du modèle global
    local_model = model_class(**model_kwargs).to(device)
    local_model.load_state_dict(copy.deepcopy(global_state_dict))

    # Loader local
    ds_local = ShapeNetDataset(df_client, PLY_DIR, augment=True)
    if len(ds_local) == 0:
        return copy.deepcopy(global_state_dict), 0
    loader_local = PyGDataLoader(ds_local, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(local_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    local_model.train()
    for _ in range(local_epochs):
        for batch in loader_local:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = criterion(local_model(batch), batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
            optimizer.step()

    return local_model.state_dict(), len(ds_local)


print('✅ FedAvg manuel implémenté')

In [ ]:
# ─── Pipeline de Fédération Horizontale ──────────────────────────────────────

def federated_training(model_class, client_dfs: dict, n_rounds: int = NUM_ROUNDS,
                       ckpt_name: str = 'fed', config_tag: str = 'HFL',
                       device=DEVICE, **model_kwargs) -> tuple:
    """
    Pipeline de fédération horizontale.

    À chaque ronde :
    1. Le serveur diffuse une copie du modèle global à chaque client
    2. Chaque client s'entraîne localement et renvoie son state_dict
    3. Le serveur applique FedAvg
    4. Évaluation du modèle global sur val set
    """
    # Initialisation du modèle global
    global_model = model_class(**model_kwargs).to(device)
    global_sd    = copy.deepcopy(global_model.state_dict())

    history = {'round': [], 'val_acc': [], 'val_loss': []}
    best_val_acc = 0.0
    criterion    = nn.CrossEntropyLoss(label_smoothing=0.1)
    ckpt_path    = CKPT_DIR / f'{ckpt_name}_best.pt'

    print(f'\n[{config_tag}] {n_rounds} rondes × {NUM_CLIENTS_H} clients × {LOCAL_EPOCHS} epochs locales')

    for rnd in range(1, n_rounds + 1):
        client_sds, client_sizes = [], []

        # Entraînement local sur chaque client
        for cid, df_c in client_dfs.items():
            sd, n_local = client_local_train(
                model_class, global_sd, df_c,
                local_epochs=LOCAL_EPOCHS, device=device, **model_kwargs
            )
            client_sds.append(sd)
            client_sizes.append(n_local)

        # Agrégation FedAvg (manuelle)
        global_sd = fed_avg(client_sds, client_sizes)

        # Évaluation du modèle global
        global_model.load_state_dict(global_sd)
        v_loss, v_acc = eval_epoch(global_model, loader_val, criterion, device)
        history['round'].append(rnd)
        history['val_acc'].append(v_acc)
        history['val_loss'].append(v_loss)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(global_sd, ckpt_path)

        if rnd % 5 == 0 or rnd == 1:
            print(f'  [Ronde {rnd:3d}/{n_rounds}] val_loss={v_loss:.4f} '
                  f'val_acc={v_acc:.4f} [best={best_val_acc:.4f}] '
                  f'| sizes={client_sizes}')

    # Charger le meilleur modèle global
    best_sd = torch.load(ckpt_path, map_location=device)
    global_model.load_state_dict(best_sd)
    print(f'  ✅ Meilleur modèle global chargé (val_acc={best_val_acc:.4f})')
    return global_model, history


print('✅ Pipeline de fédération défini')

In [ ]:
# ─── C2 : HFL IID ─────────────────────────────────────────────────────────────
print('=' * 60)
print('C2 — Fédéré Horizontal IID : TACO-Net')
print('=' * 60)
set_seed()
model_taco_c2, hist_taco_c2 = federated_training(
    TACONet, hfl_client_dfs_iid,
    n_rounds=NUM_ROUNDS, ckpt_name='taco_c2', config_tag='HFL-IID',
    in_ch=6, num_cls=NUM_CLASSES
)

print('\n' + '=' * 60)
print('C2 — Fédéré Horizontal IID : PointGCN')
print('=' * 60)
set_seed()
model_gcn_c2, hist_gcn_c2 = federated_training(
    PointGCN, hfl_client_dfs_iid,
    n_rounds=NUM_ROUNDS, ckpt_name='gcn_c2', config_tag='HFL-IID',
    in_ch=6, num_cls=NUM_CLASSES
)

In [ ]:
# ─── C3 : HFL non-IID ──────────────────────────────────────────────────────────
print('=' * 60)
print('C3 — Fédéré Horizontal non-IID : TACO-Net')
print('=' * 60)
set_seed()
model_taco_c3, hist_taco_c3 = federated_training(
    TACONet, hfl_client_dfs_niid,
    n_rounds=NUM_ROUNDS, ckpt_name='taco_c3', config_tag='HFL-NIID',
    in_ch=6, num_cls=NUM_CLASSES
)

print('\n' + '=' * 60)
print('C3 — Fédéré Horizontal non-IID : PointGCN')
print('=' * 60)
set_seed()
model_gcn_c3, hist_gcn_c3 = federated_training(
    PointGCN, hfl_client_dfs_niid,
    n_rounds=NUM_ROUNDS, ckpt_name='gcn_c3', config_tag='HFL-NIID',
    in_ch=6, num_cls=NUM_CLASSES
)

In [ ]:
# ─── Analyse HFL non-IID : performance sur classes absentes ───────────────────
# Le sujet demande d'évaluer le modèle global sur des classes dont le client
# responsable était absent lors d'une ronde.

@torch.no_grad()
def eval_by_client_domain(model, loader_te, partition, device=DEVICE):
    """Évalue la précision du modèle global par domaine client."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader_te:
        batch = batch.to(device)
        preds = model(batch).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    results = {}
    for cid, classes in partition.items():
        cls_ids = [class2idx[c] for c in classes if c in class2idx]
        mask    = np.isin(all_labels, cls_ids)
        if mask.sum() > 0:
            acc = (all_preds[mask] == all_labels[mask]).mean()
        else:
            acc = 0.0
        results[CLIENT_NAMES[cid]] = acc
    return results

print('\n📊 Analyse HFL non-IID — Précision par domaine client (modèle global TACO-Net C3) :')
domain_accs_niid = eval_by_client_domain(model_taco_c3, loader_test, HFL_PARTITION)
for dom, acc in domain_accs_niid.items():
    print(f'  {dom:<35} : {acc:.4f}')

print('\n📊 Analyse HFL IID — Précision par domaine client (modèle global TACO-Net C2) :')
domain_accs_iid = eval_by_client_domain(model_taco_c2, loader_test, HFL_PARTITION)
for dom, acc in domain_accs_iid.items():
    print(f'  {dom:<35} : {acc:.4f}')

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(NUM_CLIENTS_H)
w = 0.35
ax.bar(x_pos - w/2, domain_accs_iid.values(),  w, label='HFL IID (C2)',     color='steelblue',  alpha=0.85)
ax.bar(x_pos + w/2, domain_accs_niid.values(), w, label='HFL non-IID (C3)', color='darkorange', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels([c.split('—')[1].strip() for c in CLIENT_NAMES], rotation=15)
ax.set_ylabel('Précision')
ax.set_title('Impact non-IID sur chaque domaine client (TACO-Net)', fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'hfl_domain_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Fédération Verticale (VFL) — Partition par attributs

**Entité A (Géométrie)** : possède uniquement XYZ → produit embedding $e_A \in \mathbb{R}^d$  
**Entité B (Apparence)** : possède uniquement RGB → produit embedding $e_B \in \mathbb{R}^d$  
**Serveur** : fusionne $[e_A \| e_B]$ et calcule la perte → renvoie les gradients

**Confidentialité garantie** : aucune entité ne voit les données brutes de l'autre. Seuls les vecteurs intermédiaires circulent. L'Entité A ne peut pas reconstruire les couleurs depuis $e_A$ (la projection est irréversible sans le modèle B).

In [ ]:
# ─── VFL : encodeurs locaux (troncs partiels) ─────────────────────────────────

class VFLEncoderXYZ(nn.Module):
    """
    Entité A — Encodeur géométrique (XYZ uniquement).
    Produit un embedding e_A ∈ R^d.
    """
    def __init__(self, emb_dim: int = 256):
        super().__init__()
        # Tronc Graph CNN sur XYZ
        self.stem  = nn.Sequential(nn.Linear(3, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2))
        self.block1 = TACOBlock(64, 128)
        self.block2 = TACOBlock(128, 256)
        self.proj   = nn.Sequential(
            nn.Linear(256 * 2, emb_dim),   # max + mean pool
            nn.BatchNorm1d(emb_dim), nn.ReLU()
        )

    def forward(self, data):
        # Ne lit que les coordonnées spatiales
        x, edge_index, batch = data.pos, data.edge_index, data.batch
        h = self.stem(x)
        h = self.block1(h, edge_index)
        h = self.block2(h, edge_index)
        g = torch.cat([global_max_pool(h, batch),
                       global_mean_pool(h, batch)], dim=-1)
        return self.proj(g)   # e_A : (B, emb_dim)


class VFLEncoderRGB(nn.Module):
    """
    Entité B — Encodeur d'apparence (RGB uniquement).
    Produit un embedding e_B ∈ R^d.
    """
    def __init__(self, emb_dim: int = 256):
        super().__init__()
        # Tronc GCN sur RGB
        self.stem  = nn.Sequential(nn.Linear(3, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gcn1  = GCNConv(64, 128)
        self.gcn2  = GCNConv(128, 256)
        self.bn1   = nn.BatchNorm1d(128)
        self.bn2   = nn.BatchNorm1d(256)
        self.proj   = nn.Sequential(
            nn.Linear(256 * 2, emb_dim),
            nn.BatchNorm1d(emb_dim), nn.ReLU()
        )

    def forward(self, data):
        # Ne lit que les couleurs
        x, edge_index, batch = data.color, data.edge_index, data.batch
        h = self.stem(x)
        h = F.relu(self.bn1(self.gcn1(h, edge_index)))
        h = F.relu(self.bn2(self.gcn2(h, edge_index)))
        g = torch.cat([global_max_pool(h, batch),
                       global_mean_pool(h, batch)], dim=-1)
        return self.proj(g)   # e_B : (B, emb_dim)


class VFLServerFusion(nn.Module):
    """
    Couche de fusion côté serveur.
    Concatène [e_A || e_B] et produit les logits finaux.

    Justification confidentialité :
    - Le serveur ne voit jamais les données brutes (XYZ ou RGB)
    - e_A et e_B sont des projections denses — irréversibles sans accès au réseau source
    - La rétropropagation envoie uniquement dL/de_A et dL/de_B,
      pas les données brutes
    """
    def __init__(self, emb_dim: int = 256, num_cls: int = NUM_CLASSES):
        super().__init__()
        self.fusion_head = nn.Sequential(
            nn.Linear(emb_dim * 2, 512),
            nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_cls)
        )

    def forward(self, e_A, e_B):
        fused = torch.cat([e_A, e_B], dim=-1)   # (B, 2*emb_dim)
        return self.fusion_head(fused)


print('✅ Modules VFL définis (Entité A, Entité B, Serveur Fusion)')

In [ ]:
# ─── Entraînement VFL (split learning) ───────────────────────────────────────

def train_vfl(loader_tr, loader_va, epochs=EPOCHS_C,
              emb_dim=256, ckpt_name='vfl', device=DEVICE):
    """
    Entraînement VFL en split learning :
    1. Entité A encode XYZ → e_A (avec grad)
    2. Entité B encode RGB → e_B (avec grad)
    3. Serveur fusionne, calcule la loss, backprop
    4. Gradients dL/de_A et dL/de_B retournent aux entités

    Note : dans cette simulation, les trois composants tournent sur le même GPU
    (il n'y a pas de réseau physique entre entités). En production, e_A et e_B
    seraient sérialisés et envoyés au serveur.
    """
    enc_A  = VFLEncoderXYZ(emb_dim).to(device)
    enc_B  = VFLEncoderRGB(emb_dim).to(device)
    server = VFLServerFusion(emb_dim, NUM_CLASSES).to(device)

    opt_A  = torch.optim.AdamW(enc_A.parameters(),  lr=LR, weight_decay=WEIGHT_DECAY)
    opt_B  = torch.optim.AdamW(enc_B.parameters(),  lr=LR, weight_decay=WEIGHT_DECAY)
    opt_S  = torch.optim.AdamW(server.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched_A = torch.optim.lr_scheduler.CosineAnnealingLR(opt_A, T_max=epochs)
    sched_B = torch.optim.lr_scheduler.CosineAnnealingLR(opt_B, T_max=epochs)
    sched_S = torch.optim.lr_scheduler.CosineAnnealingLR(opt_S, T_max=epochs)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    history   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    ckpt_path = CKPT_DIR / f'{ckpt_name}_best.pt'

    for epoch in range(1, epochs + 1):
        enc_A.train(); enc_B.train(); server.train()
        t_loss, correct, total = 0.0, 0, 0

        for batch in loader_tr:
            batch = batch.to(device)

            # ── Encodage local (chaque entité traite ses seules données) ──
            e_A = enc_A(batch)   # Entité A : XYZ uniquement
            e_B = enc_B(batch)   # Entité B : RGB uniquement

            # ── Serveur : fusion + prédiction + loss ──
            logits = server(e_A, e_B)
            loss   = criterion(logits, batch.y)

            # ── Backprop : les gradients remontent vers A et B ──
            opt_A.zero_grad(); opt_B.zero_grad(); opt_S.zero_grad()
            loss.backward()
            for opt in [opt_A, opt_B, opt_S]:
                torch.nn.utils.clip_grad_norm_(
                    list(enc_A.parameters()) + list(enc_B.parameters()) + list(server.parameters()),
                    1.0
                )
                opt.step()

            t_loss  += loss.item() * batch.num_graphs
            correct += (logits.argmax(1) == batch.y).sum().item()
            total   += batch.num_graphs

        # Évaluation
        enc_A.eval(); enc_B.eval(); server.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in loader_va:
                batch  = batch.to(device)
                logits = server(enc_A(batch), enc_B(batch))
                loss   = criterion(logits, batch.y)
                v_loss    += loss.item() * batch.num_graphs
                v_correct += (logits.argmax(1) == batch.y).sum().item()
                v_total   += batch.num_graphs

        t_acc = correct  / total
        v_acc = v_correct / v_total
        history['train_loss'].append(t_loss / total)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss / v_total)
        history['val_acc'].append(v_acc)

        for sched in [sched_A, sched_B, sched_S]: sched.step()

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save({'enc_A': enc_A.state_dict(),
                        'enc_B': enc_B.state_dict(),
                        'server': server.state_dict()}, ckpt_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  [VFL Ep {epoch:3d}/{epochs}] '
                  f'train_acc={t_acc:.4f} val_acc={v_acc:.4f} [best={best_val_acc:.4f}]')

    # Charger le meilleur état
    best = torch.load(ckpt_path, map_location=device)
    enc_A.load_state_dict(best['enc_A'])
    enc_B.load_state_dict(best['enc_B'])
    server.load_state_dict(best['server'])
    print(f'  ✅ VFL — meilleur modèle chargé (val_acc={best_val_acc:.4f})')
    return enc_A, enc_B, server, history


print('=' * 60)
print('VFL — Fédération Verticale : TACO-Net + PointGCN encodeurs')
print('=' * 60)
set_seed()
enc_A_vfl, enc_B_vfl, server_vfl, hist_vfl = train_vfl(
    loader_train, loader_val, epochs=EPOCHS_C, ckpt_name='vfl_taco'
)

# Évaluation VFL sur test set
@torch.no_grad()
def eval_vfl(enc_A, enc_B, server, loader_te, device=DEVICE):
    enc_A.eval(); enc_B.eval(); server.eval()
    correct, total = 0, 0
    for batch in loader_te:
        batch  = batch.to(device)
        logits = server(enc_A(batch), enc_B(batch))
        correct += (logits.argmax(1) == batch.y).sum().item()
        total   += batch.num_graphs
    return correct / total

vfl_test_acc = eval_vfl(enc_A_vfl, enc_B_vfl, server_vfl, loader_test)
print(f'\n📊 VFL Test Accuracy : {vfl_test_acc:.4f}')

---
## 8. Partie 3 — Distillation de Connaissances

**Enseignant** : modèle centralisé C1 (full size)  
**Élève** : modèle allégé de la même famille (≥ 4× moins de paramètres)

$$\mathcal{L} = (1-\alpha)\mathcal{L}_{CE}(\hat{y}, y) + \alpha \tau^2 \mathcal{D}_{KL}\left(\sigma\!\left(\frac{z_s}{\tau}\right) \| \sigma\!\left(\frac{z_t}{\tau}\right)\right)$$

In [ ]:
# ─── Modèles élèves (allégés, ≥ 4× moins de params) ─────────────────────────

class TACONetSmall(nn.Module):
    """
    TACO-Net allégé pour distillation.
    Mêmes blocs mais widths réduites : [16, 32, 64] vs [64, 128, 256].
    """
    def __init__(self, in_ch: int = 6, num_cls: int = NUM_CLASSES,
                 widths: list = [16, 32, 64]):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_ch, widths[0]),
            nn.BatchNorm1d(widths[0]), nn.LeakyReLU(0.2)
        )
        self.blocks = nn.ModuleList()
        dims = [widths[0]] + widths
        for i in range(len(widths)):
            self.blocks.append(TACOBlock(dims[i], dims[i+1]))
        total_ch = widths[0] + sum(widths)
        self.fusion = nn.Sequential(
            nn.Linear(total_ch, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2)
        )
        self.head = nn.Sequential(
            nn.Linear(256, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Dropout(0.3), nn.Linear(64, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        skips = [h]
        for block in self.blocks:
            h = block(h, edge_index)
            skips.append(h)
        multi = torch.cat(skips, dim=-1)
        feat  = self.fusion(multi)
        g = torch.cat([global_mean_pool(feat, batch),
                       global_max_pool(feat, batch)], dim=-1)
        return self.head(g)


class PointGCNSmall(nn.Module):
    """PointGCN allégé : hidden=32 vs 128."""
    def __init__(self, in_ch: int = 6, num_cls: int = NUM_CLASSES, hidden: int = 32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_ch, hidden), nn.BatchNorm1d(hidden), nn.ReLU()
        )
        self.gcn1 = GCNConv(hidden,     hidden)
        self.gcn2 = GCNConv(hidden,     hidden * 2)
        self.gcn3 = GCNConv(hidden * 2, hidden * 4)
        self.bn1  = nn.BatchNorm1d(hidden)
        self.bn2  = nn.BatchNorm1d(hidden * 2)
        self.bn3  = nn.BatchNorm1d(hidden * 4)
        self.skip2 = nn.Linear(hidden, hidden * 2)
        self.head  = nn.Sequential(
            nn.Linear(hidden * 4 * 2, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        h = F.relu(self.bn1(self.gcn1(h, edge_index))) + h
        h = F.relu(self.bn2(self.gcn2(h, edge_index))) + self.skip2(h)
        h = F.relu(self.bn3(self.gcn3(h, edge_index)))
        g = torch.cat([global_max_pool(h, batch), global_mean_pool(h, batch)], dim=-1)
        return self.head(g)


# Vérification ratio de compression
student_taco = TACONetSmall(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
student_gcn  = PointGCNSmall(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
n_taco_s = sum(p.numel() for p in student_taco.parameters() if p.requires_grad)
n_gcn_s  = sum(p.numel() for p in student_gcn.parameters()  if p.requires_grad)

print(f'Enseignant TACO-Net : {n_params_taco:>10,} paramètres')
print(f'Élève    TACO-Net  : {n_taco_s:>10,} paramètres  — ratio {n_params_taco/n_taco_s:.1f}×')
print(f'Enseignant PointGCN : {n_params_gcn:>10,} paramètres')
print(f'Élève    PointGCN  : {n_gcn_s:>10,} paramètres  — ratio {n_params_gcn/n_gcn_s:.1f}×')
assert n_params_taco / n_taco_s >= 4, 'Élève TACO trop grand !'
assert n_params_gcn  / n_gcn_s  >= 4, 'Élève GCN trop grand !'

In [ ]:
# ─── Perte de distillation ────────────────────────────────────────────────────

def distillation_loss(logits_student, logits_teacher, labels,
                      alpha: float = ALPHA, tau: float = TEMPERATURE):
    """
    L = (1 - alpha) * CE(student_hard, labels)
      + alpha * tau^2 * KL(sigma(z_s/tau) || sigma(z_t/tau))

    Les logits enseignant sont détachés du graphe de calcul.
    """
    # Les logits de l'enseignant sont détachés (pas de grad pour l'enseignant)
    logits_teacher = logits_teacher.detach()

    # Perte hard label
    loss_ce = F.cross_entropy(logits_student, labels)

    # Perte KL (soft targets)
    soft_student = F.log_softmax(logits_student / tau, dim=-1)
    soft_teacher = F.softmax(logits_teacher    / tau, dim=-1)
    loss_kl      = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (tau ** 2)

    return (1.0 - alpha) * loss_ce + alpha * loss_kl


def train_kd(teacher, student, loader_tr, loader_va,
             epochs=EPOCHS_KD, alpha=ALPHA, tau=TEMPERATURE,
             ckpt_name='kd', device=DEVICE, verbose=True):
    """
    Entraînement par distillation de connaissances.
    L'enseignant est figé (eval mode). Seul l'élève est mis à jour.
    """
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False  # Enseignant figé

    optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    ckpt_path = CKPT_DIR / f'{ckpt_name}_best.pt'

    for epoch in range(1, epochs + 1):
        student.train()
        t_loss, correct, total = 0.0, 0, 0

        for batch in loader_tr:
            batch = batch.to(device)
            with torch.no_grad():
                logits_t = teacher(batch)
            logits_s = student(batch)
            loss = distillation_loss(logits_s, logits_t, batch.y, alpha, tau)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()

            t_loss  += loss.item() * batch.num_graphs
            correct += (logits_s.argmax(1) == batch.y).sum().item()
            total   += batch.num_graphs

        v_loss, v_acc = eval_epoch(student, loader_va, criterion, device)
        scheduler.step()

        history['train_loss'].append(t_loss / total)
        history['train_acc'].append(correct / total)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(student.state_dict(), ckpt_path)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  [KD Ep {epoch:3d}/{epochs}] α={alpha:.2f} τ={tau:.1f} '
                  f'train_acc={correct/total:.4f} val_acc={v_acc:.4f} [best={best_val_acc:.4f}]')

    student.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'  ✅ KD — meilleur élève chargé (val_acc={best_val_acc:.4f})')
    return history


print('✅ Distillation de connaissances définie')

In [ ]:
# ─── C4 : Distillation ────────────────────────────────────────────────────────
print('=' * 60)
print('C4 — Distillation : TACO-Net (enseignant → élève)')
print('=' * 60)
set_seed()
student_taco = TACONetSmall(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
hist_kd_taco = train_kd(
    model_taco_c1, student_taco, loader_train, loader_val,
    epochs=EPOCHS_KD, alpha=ALPHA, tau=TEMPERATURE, ckpt_name='kd_taco'
)

print('\n' + '=' * 60)
print('C4 — Distillation : PointGCN (enseignant → élève)')
print('=' * 60)
set_seed()
student_gcn = PointGCNSmall(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
hist_kd_gcn = train_kd(
    model_gcn_c1, student_gcn, loader_train, loader_val,
    epochs=EPOCHS_KD, alpha=ALPHA, tau=TEMPERATURE, ckpt_name='kd_gcn'
)

In [ ]:
# ─── Analyse de l'impact de τ et α sur la distillation ───────────────────────

TAU_GRID   = [1.0, 2.0, 4.0, 8.0]
ALPHA_GRID = [0.3, 0.5, 0.7, 0.9]
ABLATION_EPOCHS = 15   # Moins d'epochs pour l'ablation

ablation_results = []

print('\nAblation τ × α (TACO-Net, 15 epochs) :')
for tau in TAU_GRID:
    for alpha in ALPHA_GRID:
        set_seed()
        s = TACONetSmall(in_ch=6, num_cls=NUM_CLASSES).to(DEVICE)
        h = train_kd(model_taco_c1, s, loader_train, loader_val,
                     epochs=ABLATION_EPOCHS, alpha=alpha, tau=tau,
                     ckpt_name=f'kd_abl_tau{tau}_a{alpha}', verbose=False)
        g_acc, _, _, _ = evaluate_full(s, loader_test)
        ablation_results.append({'tau': tau, 'alpha': alpha, 'test_acc': g_acc})
        print(f'  τ={tau:.1f}  α={alpha:.1f}  → test_acc={g_acc:.4f}')

df_ablation = pd.DataFrame(ablation_results)

# Heatmap τ × α
pivot = df_ablation.pivot(index='tau', columns='alpha', values='test_acc')
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Test Accuracy'})
ax.set_title('Impact de τ et α sur la distillation (TACO-Net)', fontweight='bold', fontsize=12)
ax.set_xlabel('α (pondération KL)')
ax.set_ylabel('τ (température)')
plt.tight_layout()
plt.savefig(ROOT / 'kd_ablation_tau_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Comparaison & Analyse Finale

In [ ]:
# ─── Évaluation complète de toutes les configurations ─────────────────────────

configs = [
    ('C1 — Baseline',     model_taco_c1,  model_gcn_c1),
    ('C2 — HFL IID',      model_taco_c2,  model_gcn_c2),
    ('C3 — HFL non-IID',  model_taco_c3,  model_gcn_c3),
    ('C4 — Distillation', student_taco,   student_gcn),
]

rows = []
for config_name, m_taco, m_gcn in configs:
    n_t = sum(p.numel() for p in m_taco.parameters() if p.requires_grad)
    n_g = sum(p.numel() for p in m_gcn.parameters()  if p.requires_grad)
    g_t, m_t, _, _ = evaluate_full(m_taco, loader_test)
    g_g, m_g, _, _ = evaluate_full(m_gcn,  loader_test)
    rows.append({
        'Config':          config_name,
        'TACO Acc (glob)': round(g_t, 4),
        'TACO Acc (cls)':  round(m_t, 4),
        'TACO Params':     n_t,
        'GCN Acc (glob)':  round(g_g, 4),
        'GCN Acc (cls)':   round(m_g, 4),
        'GCN Params':      n_g,
    })

df_results = pd.DataFrame(rows)
print('\n📊 Tableau de comparaison final :')
print(df_results.to_string(index=False))

In [ ]:
# ─── Courbes de convergence toutes configurations (TACO-Net) ──────────────────

# Adapter les histoires HFL (rounds → epochs)
def adapt_fed_history(hist_fed):
    return {
        'train_loss': [0.0] * len(hist_fed['val_loss']),  # non disponible en fédéré
        'train_acc':  [0.0] * len(hist_fed['val_acc']),
        'val_loss':   hist_fed['val_loss'],
        'val_acc':    hist_fed['val_acc'],
    }

plot_convergence_curves(
    histories={
        'C1 Centralisé':    hist_taco_c1,
        'C2 HFL IID':       adapt_fed_history(hist_taco_c2),
        'C3 HFL non-IID':   adapt_fed_history(hist_taco_c3),
        'C4 Distillation':  hist_kd_taco,
    },
    title='TACO-Net — Courbes de Convergence (4 configurations)',
    save_path=ROOT / 'curves_taco_all_configs.png'
)

plot_convergence_curves(
    histories={
        'C1 Centralisé':   hist_gcn_c1,
        'C2 HFL IID':      adapt_fed_history(hist_gcn_c2),
        'C3 HFL non-IID':  adapt_fed_history(hist_gcn_c3),
        'C4 Distillation': hist_kd_gcn,
    },
    title='PointGCN — Courbes de Convergence (4 configurations)',
    save_path=ROOT / 'curves_gcn_all_configs.png'
)

In [ ]:
# ─── Visualisation finale : barplot comparatif ────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Comparaison des 4 Configurations × 2 Architectures', fontsize=14, fontweight='bold')

config_names = [r['Config'].split('—')[0].strip() for r in rows]
x = np.arange(len(rows))
w = 0.35

taco_glob = [r['TACO Acc (glob)'] for r in rows]
gcn_glob  = [r['GCN Acc (glob)']  for r in rows]
taco_cls  = [r['TACO Acc (cls)']  for r in rows]
gcn_cls   = [r['GCN Acc (cls)']   for r in rows]

axes[0].bar(x - w/2, taco_glob, w, label='TACO-Net', color='steelblue',   alpha=0.85)
axes[0].bar(x + w/2, gcn_glob,  w, label='PointGCN', color='darkorange',  alpha=0.85)
axes[0].set_title('Précision Globale (test set)',    fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(config_names, rotation=15)
axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)
for i, (a, b) in enumerate(zip(taco_glob, gcn_glob)):
    axes[0].text(i - w/2, a + 0.003, f'{a:.3f}', ha='center', fontsize=8, fontweight='bold')
    axes[0].text(i + w/2, b + 0.003, f'{b:.3f}', ha='center', fontsize=8, fontweight='bold')

axes[1].bar(x - w/2, taco_cls, w, label='TACO-Net', color='steelblue',  alpha=0.85)
axes[1].bar(x + w/2, gcn_cls,  w, label='PointGCN', color='darkorange', alpha=0.85)
axes[1].set_title('Précision Moyenne par Classe (test set)', fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(config_names, rotation=15)
axes[1].set_ylabel('Mean Class Accuracy'); axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)
for i, (a, b) in enumerate(zip(taco_cls, gcn_cls)):
    axes[1].text(i - w/2, a + 0.003, f'{a:.3f}', ha='center', fontsize=8, fontweight='bold')
    axes[1].text(i + w/2, b + 0.003, f'{b:.3f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(ROOT / 'final_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

print('\n✅ Analyse finale terminée')
print('\nFichiers générés dans /kaggle/working/ :')
for f in sorted(Path('/kaggle/working').glob('*.png')):
    print(f'  {f.name}')

In [ ]:
# ─── Sauvegarde du tableau final en CSV ──────────────────────────────────────
df_results.to_csv(ROOT / 'results_final.csv', index=False)

# ─── Résumé exécutif ─────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('RÉSUMÉ EXÉCUTIF')
print('=' * 65)
print(f'\nArchitecture 1 : TACO-Net ({n_params_taco:,} params)')
print(f'  C1 Baseline    : {df_results.loc[0, "TACO Acc (glob)"]:.4f} (glob) | {df_results.loc[0, "TACO Acc (cls)"]:.4f} (cls)')
print(f'  C2 HFL IID     : {df_results.loc[1, "TACO Acc (glob)"]:.4f} (glob) | {df_results.loc[1, "TACO Acc (cls)"]:.4f} (cls)')
print(f'  C3 HFL non-IID : {df_results.loc[2, "TACO Acc (glob)"]:.4f} (glob) | {df_results.loc[2, "TACO Acc (cls)"]:.4f} (cls)')
print(f'  C4 Distillation: {df_results.loc[3, "TACO Acc (glob)"]:.4f} (glob) | {df_results.loc[3, "TACO Acc (cls)"]:.4f} (cls) [{n_taco_s:,} params]')

print(f'\nArchitecture 2 : PointGCN ({n_params_gcn:,} params)')
print(f'  C1 Baseline    : {df_results.loc[0, "GCN Acc (glob)"]:.4f} (glob) | {df_results.loc[0, "GCN Acc (cls)"]:.4f} (cls)')
print(f'  C2 HFL IID     : {df_results.loc[1, "GCN Acc (glob)"]:.4f} (glob) | {df_results.loc[1, "GCN Acc (cls)"]:.4f} (cls)')
print(f'  C3 HFL non-IID : {df_results.loc[2, "GCN Acc (glob)"]:.4f} (glob) | {df_results.loc[2, "GCN Acc (cls)"]:.4f} (cls)')
print(f'  C4 Distillation: {df_results.loc[3, "GCN Acc (glob)"]:.4f} (glob) | {df_results.loc[3, "GCN Acc (cls)"]:.4f} (cls) [{n_gcn_s:,} params]')

print(f'\nVFL Test Accuracy : {vfl_test_acc:.4f}')
print('\n✅ Notebook complet exécuté avec succès !')